# Clip Embedding Inventory

This notebook reads the locally generated V-JEPA clip embedding caches and builds a dataframe with one row per segment.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

cache_root = Path("../data/vjepa_clip_embedding_experiments/records")
cache_paths = sorted(cache_root.glob("*.npz"))

if not cache_paths:
    raise FileNotFoundError(f"No clip embedding caches found under {cache_root.resolve()}")

rows = []
for cache_path in cache_paths:
    cache = np.load(cache_path, allow_pickle=False)
    segment_record_names = cache["segment_record_names"]
    segment_ids = cache["segment_ids"]
    segment_labels = cache["segment_labels"]
    segment_label_text = cache["segment_label_text"]
    segment_rhythms = cache["segment_rhythms"]
    segment_num_frames = cache["segment_num_frames"]
    segment_num_clips = cache["segment_num_clips"]
    
    clip_segment_ids = cache["clip_segment_ids"]
    clip_indices = cache["clip_indices"]
    clip_start_frames = cache["clip_start_frames"]
    clip_end_frames = cache["clip_end_frames"]
    embeddings = cache["embeddings"]

    clip_counts_from_rows = pd.Series(clip_segment_ids).value_counts().to_dict()
    clip_index_min = pd.DataFrame({"segment_id": clip_segment_ids, "clip_index": clip_indices}).groupby("segment_id")["clip_index"].min().to_dict()
    clip_index_max = pd.DataFrame({"segment_id": clip_segment_ids, "clip_index": clip_indices}).groupby("segment_id")["clip_index"].max().to_dict()
    clip_start_min = pd.DataFrame({"segment_id": clip_segment_ids, "clip_start_frame": clip_start_frames}).groupby("segment_id")["clip_start_frame"].min().to_dict()
    clip_end_max = pd.DataFrame({"segment_id": clip_segment_ids, "clip_end_frame": clip_end_frames}).groupby("segment_id")["clip_end_frame"].max().to_dict()

    for idx in range(len(segment_ids)):
        segment_id = int(segment_ids[idx])
        rows.append(
            {
                "record_name": str(segment_record_names[idx]),
                "segment_id": segment_id,
                "segment_key": f"{segment_record_names[idx]}_{segment_id:04d}",
                "label_id": int(segment_labels[idx]),
                "label_text": str(segment_label_text[idx]),
                "rhythm": str(segment_rhythms[idx]),
                "num_frames": int(segment_num_frames[idx]),
                "num_clips": int(segment_num_clips[idx]),
                "num_clips_from_rows": int(clip_counts_from_rows.get(segment_id, 0)),
                "first_clip_index": int(clip_index_min.get(segment_id, -1)),
                "last_clip_index": int(clip_index_max.get(segment_id, -1)),
                "first_clip_start_frame": int(clip_start_min.get(segment_id, -1)),
                "last_clip_end_frame": int(clip_end_max.get(segment_id, -1)),
                "embedding_dim": int(embeddings.shape[1]),
                "cache_file": cache_path.name,
            }
        )

segments_df = pd.DataFrame(rows).sort_values(["record_name", "segment_id"]).reset_index(drop=True)

print(f"Loaded {len(cache_paths)} record cache files from {cache_root.resolve()}")
print(f"Built segment dataframe with {len(segments_df)} rows")


Loaded 45 record cache files from /Users/kunzhan/github/kun1887/CV_group_J/src/data/vjepa_clip_embedding_experiments/records
Built segment dataframe with 1216 rows


In [15]:
segments_df.describe()

,segment_id,label_id,num_frames,num_clips,num_clips_from_rows,first_clip_index,last_clip_index,first_clip_start_frame,last_clip_end_frame,embedding_dim
count,1216.000000,1216.000000,1216.000000,1216.000000,1216.000000,1216.0,1216.000000,1216.0,1216.000000,1216.0
mean,47.286184,0.595395,67.302632,8.395559,8.395559,0.0,7.395559,0.0,67.302632,1024.0
std,47.947437,0.491017,242.981001,30.213952,30.213952,0.0,30.213952,0.0,242.981001,0.0
min,0.000000,0.000000,1.000000,1.000000,1.000000,0.0,0.000000,0.0,1.000000,1024.0
25%,11.000000,0.000000,4.000000,1.000000,1.000000,0.0,0.000000,0.0,4.000000,1024.0
50%,29.000000,1.000000,8.000000,1.000000,1.000000,0.0,0.000000,0.0,8.000000,1024.0
75%,72.000000,1.000000,24.000000,2.000000,2.000000,0.0,1.000000,0.0,24.000000,1024.0
max,206.000000,1.000000,1806.000000,225.000000,225.000000,0.0,224.000000,0.0,1806.000000,1024.0


In [13]:
segments_df["num_clips"].value_counts().sort_index()

num_clips
1      846
2       67
3       40
4       27
5       18
      ... 
137      1
142      1
156      1
203      1
225     18
Name: count, Length: 61, dtype: int64

In [16]:
display(segments_df.groupby("label_text").agg(num_segments=("segment_id", "size"), mean_num_clips=("num_clips", "mean"), max_num_clips=("num_clips", "max")))

,num_segments,mean_num_clips,max_num_clips
label_text,,,
normal rhythm,492,15.136179,225
not-normal rhythm,724,3.814917,225
